# Winner Exato + Cascata BI-RADS 2 vs 3 (best-shot)

Base: winner 0.84027 (BERTimbau Large 5-fold, MAX_LEN 512, Focal Loss, build_input_text, calibracao fixa).

Adicao: classificador binario BERTimbau Large para 2 vs 3, mesma 5-fold, weighted CE, override bidirecional.

Thresholds de override otimizados no OOF (apenas 2 escalares, sem risco de overfit).

Meta: 0.85+.

Tempo estimado: aproximadamente 5 horas em T4 (5 folds 7-classes aproximadamente 40min + 5 folds binarios aproximadamente 20min por fold).

In [ ]:
import os, re, json, random, warnings, gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

MAX_LEN      = 512
BATCH_SIZE   = 8
EPOCHS_7C    = 5
EPOCHS_BIN   = 4
LR           = 2e-5
LR_BIN       = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS      = 5
SEED         = 42
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25
NUM_CLASSES  = 7

BEST_TEMP       = 0.958
BEST_THRESHOLDS = [0.95, 1.6, 1.35, 1.0, 0.4, 1.15, 0.1]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def find_model_path():
    candidates = [
        '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
        '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
        '/kaggle/input/bert-large-portuguese-cased',
        '/kaggle/input/bertimbau-ptbr-complete',
        '/kaggle/input/bertimbau-large',
        '/kaggle/input/bertimbau',
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'config.json')):
            return c
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    for cfg_path in root.rglob('config.json'):
        try:
            with open(cfg_path) as f:
                cfg = json.load(f)
            if cfg.get('model_type') == 'bert' and cfg.get('hidden_size', 0) >= 1024:
                parent = cfg_path.parent
                has_tokenizer = any((parent / f).exists() for f in ['tokenizer.json', 'vocab.txt', 'tokenizer_config.json'])
                has_weights = any(p.suffix in ('.bin', '.safetensors') for p in parent.iterdir())
                if has_tokenizer and has_weights:
                    return str(parent)
        except Exception:
            continue
    return None

MODEL_PATH = find_model_path()
assert MODEL_PATH is not None, 'BERTimbau Large model not found in /kaggle/input'
print('MODEL_PATH:', MODEL_PATH)

In [ ]:
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def clean_text(t):
    if not isinstance(t, str):
        return ''
    t = t.lower()
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def extract_section(text, markers, next_markers=None):
    if not isinstance(text, str):
        return ''
    low = text.lower()
    start = -1
    used_marker_len = 0
    for m in markers:
        idx = low.find(m)
        if idx != -1:
            start = idx
            used_marker_len = len(m)
            break
    if start == -1:
        return ''
    section_start = start + used_marker_len
    section_end = len(text)
    if next_markers:
        for nm in next_markers:
            nidx = low.find(nm, section_start)
            if nidx != -1 and nidx < section_end:
                section_end = nidx
    return text[section_start:section_end].strip()

def build_input_text(report):
    if not isinstance(report, str):
        return ''
    all_markers = INDICACAO_MARKERS + ACHADOS_MARKERS + COMPARATIVA_MARKERS
    indicacao = extract_section(report, INDICACAO_MARKERS, next_markers=ACHADOS_MARKERS + COMPARATIVA_MARKERS)
    achados = extract_section(report, ACHADOS_MARKERS, next_markers=COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS, next_markers=None)
    parts = []
    if indicacao:
        parts.append('indicação clínica: ' + clean_text(indicacao))
    if achados:
        parts.append('achados: ' + clean_text(achados))
    if comparativa:
        parts.append('análise comparativa: ' + clean_text(comparativa))
    if not parts:
        return clean_text(report)
    return ' [SEP] '.join(parts)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
train_texts  = [build_input_text(t) for t in train_df['report'].fillna('').tolist()]
train_labels = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].fillna('').tolist()]
print('Train:', len(train_texts), 'Test:', len(test_texts))
print('Class dist:', pd.Series(train_labels).value_counts().sort_index().to_dict())
print('Classe 2 + 3:', sum(1 for l in train_labels if l in (2, 3)))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class SPRDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx] if self.texts[idx] else ''
        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    n = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)
        with autocast():
            if token_type_ids is not None:
                out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * input_ids.size(0)
        n += input_ids.size(0)
    return total_loss / max(n, 1)

@torch.no_grad()
def predict(model, loader, return_labels=False):
    model.eval()
    all_probs = []
    all_labels = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)
        with autocast():
            if token_type_ids is not None:
                out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = F.softmax(out.logits.float(), dim=-1)
        all_probs.append(probs.cpu().numpy())
        if return_labels and 'labels' in batch:
            all_labels.append(batch['labels'].numpy())
    probs = np.concatenate(all_probs, axis=0)
    if return_labels and all_labels:
        return probs, np.concatenate(all_labels, axis=0)
    return probs

def compute_f1(probs, labels):
    return f1_score(labels, probs.argmax(axis=1), average='macro')

def temperature_scale(probs, temp):
    logits = np.log(probs + 1e-10)
    scaled = logits / temp
    exp_s = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_s / exp_s.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    return (probs * np.array(thresholds)).argmax(axis=1)

def build_model(num_labels):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=num_labels,
        local_files_only=True,
    )
    return model.to(device)

def cleanup(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
set_seed(SEED)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels_arr = np.array(train_labels)

oof_7c = np.zeros((len(train_texts), NUM_CLASSES), dtype=np.float32)
test_7c = np.zeros((len(test_texts), NUM_CLASSES), dtype=np.float32)

fold_splits = list(skf.split(train_texts, labels_arr))

test_dataset = SPRDataset(test_texts, None, tokenizer, MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'===== Phase A Fold {fold+1}/{N_FOLDS} =====')
    set_seed(SEED + fold)
    tr_texts_f = [train_texts[i] for i in tr_idx]
    tr_labels_f = [train_labels[i] for i in tr_idx]
    va_texts_f = [train_texts[i] for i in va_idx]
    va_labels_f = [train_labels[i] for i in va_idx]

    tr_ds = SPRDataset(tr_texts_f, tr_labels_f, tokenizer, MAX_LEN)
    va_ds = SPRDataset(va_texts_f, va_labels_f, tokenizer, MAX_LEN)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(NUM_CLASSES)
    no_decay = ['bias', 'LayerNorm.weight']
    grouped = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(grouped, lr=LR)
    total_steps = len(tr_loader) * EPOCHS_7C
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    loss_fn = FocalLoss(FOCAL_ALPHA, FOCAL_GAMMA)
    scaler = GradScaler()

    best_f1 = -1.0
    best_val_probs = None
    best_test_probs = None
    for epoch in range(EPOCHS_7C):
        tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, loss_fn, scaler)
        val_probs, val_lbls = predict(model, va_loader, return_labels=True)
        val_f1 = compute_f1(val_probs, val_lbls)
        print(f'  epoch {epoch+1} loss {tr_loss:.4f} val_f1 {val_f1:.5f}')
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_val_probs = val_probs
            best_test_probs = predict(model, test_loader)

    oof_7c[va_idx] = best_val_probs
    test_7c += best_test_probs / N_FOLDS
    print(f'  fold best val_f1 {best_f1:.5f}')
    cleanup(model, optimizer, scheduler, scaler, tr_loader, va_loader, tr_ds, va_ds)

oof_7c_cal = temperature_scale(oof_7c, BEST_TEMP)
oof_7c_preds = apply_thresholds(oof_7c_cal, BEST_THRESHOLDS)
oof_f1_baseline = f1_score(labels_arr, oof_7c_preds, average='macro')
print(f'Phase A OOF F1 (winner calibration): {oof_f1_baseline:.5f}')

In [ ]:
mask_23 = np.isin(labels_arr, [2, 3])
bin_labels_full = np.where(labels_arr == 3, 1, 0)

oof_bin = np.zeros(len(train_texts), dtype=np.float32)
test_bin = np.zeros(len(test_texts), dtype=np.float32)
bin_oof_mask = np.zeros(len(train_texts), dtype=bool)

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f'===== Phase B Fold {fold+1}/{N_FOLDS} =====')
    set_seed(SEED + 100 + fold)
    tr_mask_23 = mask_23[tr_idx]
    tr_idx_23 = tr_idx[tr_mask_23]

    tr_texts_b = [train_texts[i] for i in tr_idx_23]
    tr_labels_b = bin_labels_full[tr_idx_23].tolist()
    va_texts_b = [train_texts[i] for i in va_idx]
    va_labels_b = bin_labels_full[va_idx].tolist()

    class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=tr_labels_b)
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    loss_fn_bin = nn.CrossEntropyLoss(weight=weights_tensor)

    tr_ds = SPRDataset(tr_texts_b, tr_labels_b, tokenizer, MAX_LEN)
    va_ds = SPRDataset(va_texts_b, va_labels_b, tokenizer, MAX_LEN)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(2)
    no_decay = ['bias', 'LayerNorm.weight']
    grouped = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(grouped, lr=LR_BIN)
    total_steps = len(tr_loader) * EPOCHS_BIN
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = GradScaler()

    va_mask_23 = mask_23[va_idx]
    va_lbls_23 = bin_labels_full[va_idx][va_mask_23]
    best_f1 = -1.0
    best_val_probs = None
    best_test_probs = None
    for epoch in range(EPOCHS_BIN):
        def loss_closure(logits, labels):
            return loss_fn_bin(logits, labels)
        tr_loss = train_epoch(model, tr_loader, optimizer, scheduler, loss_closure, scaler)
        val_probs = predict(model, va_loader)
        val_probs_23 = val_probs[va_mask_23]
        val_f1 = f1_score(va_lbls_23, val_probs_23.argmax(axis=1), average='macro') if len(va_lbls_23) > 0 else 0.0
        print(f'  epoch {epoch+1} loss {tr_loss:.4f} val_f1(2v3) {val_f1:.5f}')
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_val_probs = val_probs
            best_test_probs = predict(model, test_loader)

    oof_bin[va_idx] = best_val_probs[:, 1]
    bin_oof_mask[va_idx] = True
    test_bin += best_test_probs[:, 1] / N_FOLDS
    print(f'  fold best val_f1(2v3) {best_f1:.5f}')
    cleanup(model, optimizer, scheduler, scaler, tr_loader, va_loader, tr_ds, va_ds)

bin_oof_preds = (oof_bin[mask_23] > 0.5).astype(int)
bin_f1 = f1_score(bin_labels_full[mask_23], bin_oof_preds, average='macro')
print(f'Phase B Binary OOF F1 (2v3 only, threshold 0.5): {bin_f1:.5f}')

In [ ]:
def apply_cascade(preds_7c, probs_7c, probs_bin, threshold_a, threshold_b):
    out = preds_7c.copy()
    sorted_idx = probs_7c.argsort(axis=1)
    second_class = sorted_idx[:, -2]
    mask_a = (preds_7c == 2) & (second_class == 3) & (probs_bin >= threshold_a)
    out[mask_a] = 3
    mask_b = (preds_7c == 3) & (second_class == 2) & (probs_bin < (1 - threshold_b))
    out[mask_b] = 2
    return out, int(mask_a.sum()), int(mask_b.sum())

print('Grid search thresholds on OOF...')
oof_7c_preds_baseline = oof_7c_preds
best_f1 = oof_f1_baseline
best_ta = 0.5
best_tb = 0.5

for ta in np.arange(0.30, 0.72, 0.02):
    for tb in np.arange(0.30, 0.72, 0.02):
        preds_new, na, nb = apply_cascade(oof_7c_preds_baseline, oof_7c_cal, oof_bin, ta, tb)
        f1 = f1_score(labels_arr, preds_new, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_ta = float(ta)
            best_tb = float(tb)

print(f'Baseline OOF F1 (no cascade): {oof_f1_baseline:.5f}')
print(f'Best cascade OOF F1:         {best_f1:.5f}')
print(f'Best threshold_a (upgrade 2 to 3): {best_ta:.3f}')
print(f'Best threshold_b (downgrade 3 to 2): {best_tb:.3f}')
print(f'Expected gain: {(best_f1 - oof_f1_baseline)*100:.3f} pp')

In [ ]:
test_7c_cal = temperature_scale(test_7c, BEST_TEMP)
test_7c_preds = apply_thresholds(test_7c_cal, BEST_THRESHOLDS)

test_preds, na, nb = apply_cascade(test_7c_preds, test_7c_cal, test_bin, best_ta, best_tb)
print(f'Test samples: {len(test_preds)}')
print(f'Overrides 2 to 3: {na}')
print(f'Overrides 3 to 2: {nb}')
print('Prediction distribution:')
print(pd.Series(test_preds).value_counts().sort_index())

In [ ]:
submission = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved:', submission.shape)
print(submission.head())